# 03 — Préparer train et validation

Notebook Google Colab pour le projet Bug Predictor. Exécuter les cellules dans l’ordre.

## Objectif et règle de séparation
Ce notebook transforme les partitions **train** et **val** en fichiers Parquet prêts pour l’étape d’entraînement. Il dérive `file_defective` de `lines`, décode `content` comme texte et conserve des métadonnées dans le même fichier uniquement pour audit et analyses de séparation. Le modèle doit utiliser exclusivement `source` comme entrée et `file_defective` comme cible. Le test est volontairement exclu de tous les exports.

In [1]:
!pip -q install pyarrow pandas

In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import hashlib, json, time
import pyarrow as pa
import pyarrow.parquet as pq

ROOT = Path('/content/drive/MyDrive/BugPredictor/defectors_v2')
RAW_DIR = ROOT / 'raw' / 'line_bug_prediction_splits' / 'time'
OUT_DIR = ROOT / 'processed' / 'time'
OUT_DIR.mkdir(parents=True, exist_ok=True)
SPLITS_TO_EXPORT = ('train', 'val')
BATCH_SIZE = 512

Mounted at /content/drive


### Définir la transformation
Les lignes sans contenu ou sans label ne sont pas silencieusement transformées en négatifs : elles sont écartées des exports et comptées dans le manifeste. Les labels non vides sont conservés comme positifs. Les métadonnées (dépôt, chemin, date, commit) ne doivent jamais être fournies au modèle.

In [3]:
OUTPUT_SCHEMA = pa.schema([
    ('row_id', pa.string()),
    ('source', pa.string()),
    ('file_defective', pa.int8()),
    ('repo', pa.string()),
    ('filepath', pa.string()),
    ('datetime', pa.string()),
    ('commit', pa.string()),
])

def as_text(value):
    if value is None:
        return None, False
    if isinstance(value, str):
        return value, False
    raw = bytes(value)
    text = raw.decode('utf-8', errors='replace')
    return text, ('\ufffd' in text)

def stringify(value):
    if value is None:
        return None
    if hasattr(value, 'isoformat'):
        return value.isoformat()
    return str(value)

def transform_split(split):
    source_path = RAW_DIR / f'{split}.parquet.gzip'
    if not source_path.exists():
        raise FileNotFoundError(f'{source_path} absent. Exécutez le notebook 01.')
    target_path = OUT_DIR / f'{split}_prepared.parquet'
    temp_path = OUT_DIR / f'{split}_prepared.parquet.part'
    pf = pq.ParquetFile(source_path)
    required = {'content', 'lines', 'repo', 'filepath', 'datetime', 'commit'}
    # Use logical top-level Arrow field names for nested Parquet List columns.
    arrow_columns = set(pf.schema_arrow.names)
    missing = required - arrow_columns
    if missing:
        raise ValueError(f'Colonnes absentes de {split}: {missing}. Colonnes Arrow: {sorted(arrow_columns)}')
    counts = {'input_rows': 0, 'written_rows': 0, 'dropped_null_content': 0, 'dropped_null_label': 0, 'utf8_replacement_rows': 0, 'positive': 0, 'negative': 0}
    writer = None
    ordinal = 0
    try:
        for batch in pf.iter_batches(batch_size=BATCH_SIZE, columns=['content','lines','repo','filepath','datetime','commit']):
            out_rows = []
            for row in batch.to_pylist():
                counts['input_rows'] += 1
                ordinal += 1
                content, labels = row['content'], row['lines']
                if content is None:
                    counts['dropped_null_content'] += 1
                    continue
                if labels is None:
                    counts['dropped_null_label'] += 1
                    continue
                source, had_replacement = as_text(content)
                counts['utf8_replacement_rows'] += int(had_replacement)
                label = int(len(labels) > 0)
                counts['positive' if label else 'negative'] += 1
                row_id = f'{split}:{ordinal}'
                out_rows.append({
                    'row_id': row_id, 'source': source, 'file_defective': label,
                    'repo': stringify(row.get('repo')), 'filepath': stringify(row.get('filepath')),
                    'datetime': stringify(row.get('datetime')), 'commit': stringify(row.get('commit'))
                })
            if out_rows:
                table = pa.Table.from_pylist(out_rows, schema=OUTPUT_SCHEMA)
                if writer is None:
                    writer = pq.ParquetWriter(temp_path, OUTPUT_SCHEMA, compression='zstd')
                writer.write_table(table)
                counts['written_rows'] += len(out_rows)
        if writer is None:
            raise RuntimeError(f'Aucune ligne exploitable dans {split}.')
    finally:
        if writer is not None:
            writer.close()
    temp_path.replace(target_path)
    return target_path, counts

In [4]:
manifest = {
    'dataset': 'Defectors v2', 'doi': '10.5281/zenodo.7708984',
    'partition_family': 'line_bug_prediction_splits/time',
    'created_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'feature_column': 'source',
    'target_column': 'file_defective',
    'target_rule': '1 if lines is non-empty; 0 if lines is empty',
    'excluded_feature_columns': ['repo', 'filepath', 'datetime', 'commit', 'row_id'],
    'test_policy': 'test was not read by this transformation and must not be used for fitting, tuning, calibration, or threshold selection',
    'exports': {}
}
for split in SPLITS_TO_EXPORT:
    output, counts = transform_split(split)
    manifest['exports'][split] = {'path': str(output), **counts}
    print(split, output, counts)
manifest_path = OUT_DIR / 'prepared_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
print('Manifeste:', manifest_path)

train /content/drive/MyDrive/BugPredictor/defectors_v2/processed/time/train_prepared.parquet {'input_rows': 185369, 'written_rows': 182680, 'dropped_null_content': 2689, 'dropped_null_label': 0, 'utf8_replacement_rows': 122, 'positive': 83139, 'negative': 99541}
val /content/drive/MyDrive/BugPredictor/defectors_v2/processed/time/val_prepared.parquet {'input_rows': 10000, 'written_rows': 9768, 'dropped_null_content': 232, 'dropped_null_label': 0, 'utf8_replacement_rows': 4, 'positive': 1246, 'negative': 8522}
Manifeste: /content/drive/MyDrive/BugPredictor/defectors_v2/processed/time/prepared_manifest.json


### Contrôle final et exemple d’usage
Le contrôle ci-dessous vérifie le schéma et les labels. Il ne calcule aucune performance. Lors d’un entraînement futur, ajustez le vectoriseur sur `train` seulement; transformez `val` avec ce vectoriseur inchangé. Préservez le test pour une évaluation finale unique, après décision d’équipe sur le découpage et les chevauchements révélés par l’audit.

In [5]:
for split in SPLITS_TO_EXPORT:
    path = OUT_DIR / f'{split}_prepared.parquet'
    pf = pq.ParquetFile(path)
    print(split, pf.metadata.num_rows, pf.schema)
    assert {'source','file_defective','repo','filepath','datetime','commit','row_id'} <= set(pf.schema.names)
    table = pq.read_table(path, columns=['file_defective'])
    labels = table.column('file_defective').to_pylist()
    assert all(value in (0, 1) for value in labels)

print('X = colonne source uniquement; y = file_defective. Les colonnes de métadonnées servent aux audits, pas aux caractéristiques du modèle.')

train 182680 <pyarrow._parquet.ParquetSchema object at 0x7b52e03338c0>
required group field_id=-1 schema {
  optional binary field_id=-1 row_id (String);
  optional binary field_id=-1 source (String);
  optional int32 field_id=-1 file_defective (Int(bitWidth=8, isSigned=true));
  optional binary field_id=-1 repo (String);
  optional binary field_id=-1 filepath (String);
  optional binary field_id=-1 datetime (String);
  optional binary field_id=-1 commit (String);
}

val 9768 <pyarrow._parquet.ParquetSchema object at 0x7b52c051ce80>
required group field_id=-1 schema {
  optional binary field_id=-1 row_id (String);
  optional binary field_id=-1 source (String);
  optional int32 field_id=-1 file_defective (Int(bitWidth=8, isSigned=true));
  optional binary field_id=-1 repo (String);
  optional binary field_id=-1 filepath (String);
  optional binary field_id=-1 datetime (String);
  optional binary field_id=-1 commit (String);
}

X = colonne source uniquement; y = file_defective. Les colon

Les exports sont stockés dans `MyDrive/BugPredictor/defectors_v2/processed/time/`. Ne publiez pas les fichiers de code source sans vérifier les licences des dépôts d’origine et les conditions d’attribution.